# Supplier Spend, Invoice Reconciliation & Weather Pipeline (Snowflake + Python)

**Author:** Rishabh Soni
Developed from a course project in the MS in Business Analytics program, UC San Diego Rady School of Management (2024)

This notebook builds an end-to-end ETL pipeline in Snowflake for a wholesale distributor's supplier data. It brings five very different sources into one warehouse and joins them into an analysis-ready table:

| Source | Format | What it holds |
|---|---|---|
| Monthly purchase orders | 41 CSV files | Order headers and line items |
| Supplier transactions | XML | Supplier invoices and payments |
| Supplier master data | PostgreSQL | Supplier names, categories, postal codes |
| US ZIP code geography | Census Gazetteer (tab-delimited) | ZIP code centroids (lat/long) |
| Daily weather | Snowflake Marketplace (Cybersyn NOAA) | Weather station readings |

**Pipeline at a glance**

1. Stage and load the monthly purchase-order CSVs into one table
2. Calculate the total value of each purchase order
3. Parse the XML supplier invoices into a table
4. Match purchase orders to invoices
5. Compare invoiced vs. quoted amounts
6. Extract supplier data from PostgreSQL and load it into Snowflake
7. Map each supplier ZIP code to its nearest NOAA weather station and pull daily high temperatures
8. Join invoices to the weather on each transaction date
9. Analysis: how often invoices differ from quotes, and whether weather is related

## 0. Setup

Credentials are read from environment variables so nothing sensitive lives in the notebook. Copy `.env.example` to `.env`, fill in your values, and put the source files in `data/` (see the README).

In [ ]:
# !pip install snowflake-connector-python psycopg2-binary pandas matplotlib python-dotenv

import os
import csv
import fnmatch
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import snowflake.connector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # fall back to variables already set in the environment

DATA_DIR = Path(os.getenv("DATA_DIR", "data"))
PO_DIR = DATA_DIR / "Monthly_PO_Data"
XML_FILE = DATA_DIR / "Supplier_Transactions_XML.xml"
GAZ_FILE = DATA_DIR / "2021_Gaz_zcta_national.txt"

In [ ]:
# Connect to Snowflake
conn = snowflake.connector.connect(
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_PASSWORD"],
    account=os.environ["SNOWFLAKE_ACCOUNT"],
)
cs = conn.cursor()

# Warehouse, database and schema used throughout
cs.execute("CREATE WAREHOUSE IF NOT EXISTS my_first_warehouse")
cs.execute("USE WAREHOUSE my_first_warehouse")
cs.execute("CREATE DATABASE IF NOT EXISTS mydb")
cs.execute("USE DATABASE mydb")
cs.execute("CREATE SCHEMA IF NOT EXISTS my_schema")
cs.execute("USE SCHEMA my_schema")

def query_df(sql):
    # Run a query and return the result as a pandas DataFrame
    cs.execute(sql)
    cols = [c[0] for c in cs.description]
    return pd.DataFrame(cs.fetchall(), columns=cols)

## 1. Load the monthly purchase orders

Every monthly CSV is uploaded to an internal stage with `PUT`, then loaded into a single table with `COPY INTO`. Dates arrive as `MM/DD/YYYY` text, so they are converted during the load. The two free-text comment columns (`$9`, `$10`) are skipped.

In [ ]:
PO_STAGE = "po_stage"
cs.execute(f"CREATE OR REPLACE STAGE {PO_STAGE}")

csv_files = sorted(f for f in os.listdir(PO_DIR) if fnmatch.fnmatch(f, "*.csv"))
for file_name in csv_files:
    cs.execute(f"PUT 'file://{(PO_DIR / file_name).resolve()}' @{PO_STAGE}")
print(f"Staged {len(csv_files)} purchase-order files")

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE purchase_orders (
    PurchaseOrderID            INT,
    SupplierID                 INT,
    OrderDate                  DATE,
    DeliveryMethodID           INT,
    ContactPersonID            INT,
    ExpectedDeliveryDate       DATE,
    SupplierReference          STRING,
    IsOrderFinalized           INT,
    LastEditedBy               INT,
    LastEditedWhen             TIMESTAMP,
    PurchaseOrderLineID        INT,
    StockItemID                INT,
    OrderedOuters              INT,
    Description                STRING,
    ReceivedOuters             INT,
    PackageTypeID              INT,
    ExpectedUnitPricePerOuter  FLOAT,
    LastReceiptDate            DATE,
    IsOrderLineFinalized       INT,
    Line_LastEditedBy          INT,
    Line_LastEditedWhen        TIMESTAMP
)
''')

cs.execute(f'''
COPY INTO purchase_orders
FROM (
    SELECT
        $1, $2,
        TO_DATE($3, 'MM/DD/YYYY'),                -- OrderDate
        $4, $5,
        TO_DATE($6, 'MM/DD/YYYY'),                -- ExpectedDeliveryDate
        $7::STRING,                               -- SupplierReference
        $8,
        $11,                                      -- LastEditedBy ($9, $10 are comments)
        TO_TIMESTAMP($12, 'MM/DD/YYYY HH24:MI'),  -- LastEditedWhen
        $13, $14, $15,
        $16::STRING,                              -- Description
        $17, $18,
        $19::FLOAT,                               -- ExpectedUnitPricePerOuter
        TO_DATE($20, 'MM/DD/YYYY'),               -- LastReceiptDate
        $21, $22,
        TO_TIMESTAMP($23, 'MM/DD/YYYY HH24:MI')   -- Line_LastEditedWhen
    FROM @{PO_STAGE}
)
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
PATTERN = '.*[.]csv[.]gz'
ON_ERROR = 'CONTINUE'
''')

cs.execute(f"REMOVE @{PO_STAGE}")  # clean up staged files
query_df("SELECT COUNT(*) AS po_lines, COUNT(DISTINCT PurchaseOrderID) AS purchase_orders FROM purchase_orders")

## 2. Calculate the value of each purchase order

`POAmount` is the sum of `ReceivedOuters × ExpectedUnitPricePerOuter` across every line of an order. It is attached back to each line.

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE purchase_orders_with_amount AS
SELECT
    po.*,
    totals.POAmount
FROM purchase_orders po
JOIN (
    SELECT PurchaseOrderID,
           SUM(ReceivedOuters * ExpectedUnitPricePerOuter) AS POAmount
    FROM purchase_orders
    GROUP BY PurchaseOrderID
) totals
  ON po.PurchaseOrderID = totals.PurchaseOrderID
''')
query_df("SELECT DISTINCT PurchaseOrderID, SupplierID, OrderDate, POAmount FROM purchase_orders_with_amount ORDER BY PurchaseOrderID LIMIT 5")

## 3. Parse the XML supplier invoices

The XML file is parsed in Python with `ElementTree` and inserted into a typed table in one batch.

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE supplier_invoices (
    SupplierTransactionID  INTEGER,
    SupplierID             INTEGER,
    TransactionTypeID      INTEGER,
    PurchaseOrderID        INTEGER,
    PaymentMethodID        INTEGER,
    SupplierInvoiceNumber  STRING,
    TransactionDate        STRING,
    AmountExcludingTax     FLOAT,
    TaxAmount              FLOAT,
    TransactionAmount      FLOAT,
    OutstandingBalance     FLOAT,
    FinalizationDate       STRING,
    IsFinalized            INTEGER,
    LastEditedBy           INTEGER,
    LastEditedWhen         STRING
)
''')

def text(row, tag):
    node = row.find(tag)
    return node.text if node is not None else None

def num(row, tag, cast=float):
    value = text(row, tag)
    return cast(value) if value not in (None, "") else None

root = ET.parse(XML_FILE).getroot()
rows = [
    (
        num(r, "SupplierTransactionID", int),
        num(r, "SupplierID", int),
        num(r, "TransactionTypeID", int),
        num(r, "PurchaseOrderID", int),
        num(r, "PaymentMethodID", int),
        text(r, "SupplierInvoiceNumber"),
        text(r, "TransactionDate"),
        num(r, "AmountExcludingTax"),
        num(r, "TaxAmount"),
        num(r, "TransactionAmount"),
        num(r, "OutstandingBalance"),
        text(r, "FinalizationDate"),
        num(r, "IsFinalized", int),
        num(r, "LastEditedBy", int),
        text(r, "LastEditedWhen"),
    )
    for r in root.findall("row")
]

cs.executemany(
    "INSERT INTO supplier_invoices VALUES (" + ", ".join(["%s"] * 15) + ")",
    rows,
)
print(f"Inserted {len(rows)} supplier transactions")

## 4. Match purchase orders to invoices

Purchase orders and invoices are joined on both `SupplierID` and `PurchaseOrderID`. Transactions without a purchase order (such as payments) drop out.

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE purchases_supplier AS
SELECT DISTINCT
    po.PurchaseOrderID,
    po.SupplierID,
    po.OrderDate,
    po.POAmount,
    inv.SupplierTransactionID,
    inv.SupplierInvoiceNumber,
    inv.TransactionDate,
    inv.AmountExcludingTax
FROM purchase_orders_with_amount po
JOIN supplier_invoices inv
  ON po.SupplierID = inv.SupplierID
 AND po.PurchaseOrderID = inv.PurchaseOrderID
''')
query_df("SELECT COUNT(*) AS matched_invoices FROM purchases_supplier")

## 5. Invoiced vs. quoted amounts

`Invoice_VS_Quoted` = invoiced amount (excluding tax) minus the purchase-order amount. A positive value means the supplier billed more than the order was worth.

*Change from the original version:* the earlier join was at line level, so each order appeared once per line item. The view now has one row per purchase order and invoice.

In [ ]:
cs.execute('''
CREATE OR REPLACE VIEW purchase_orders_and_invoices AS
SELECT
    PurchaseOrderID,
    SupplierID,
    SupplierTransactionID,
    TransactionDate,
    AmountExcludingTax,
    POAmount,
    AmountExcludingTax - POAmount AS Invoice_VS_Quoted
FROM purchases_supplier
''')
query_df("SELECT * FROM purchase_orders_and_invoices ORDER BY PurchaseOrderID LIMIT 10")

## 6. Load supplier master data from PostgreSQL

Supplier records are extracted from PostgreSQL with `psycopg2` and written to CSV. A `CREATE TABLE` statement is generated from the inferred column types, and the file is loaded through the table's own stage.

In [ ]:
SUPPLIER_CSV = DATA_DIR / "supplier_case_data.csv"

pg_conn = psycopg2.connect(
    host=os.getenv("PG_HOST", "127.0.0.1"),
    port=os.getenv("PG_PORT", "5432"),
    dbname=os.environ["PG_DATABASE"],
    user=os.environ["PG_USER"],
    password=os.environ["PG_PASSWORD"],
)
with pg_conn.cursor() as pg_cur:
    pg_cur.execute("SELECT * FROM supplier_case")
    with open(SUPPLIER_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([d[0] for d in pg_cur.description])
        writer.writerows(pg_cur.fetchall())
pg_conn.close()

def create_table_sql(csv_file, table_name):
    # Build a CREATE TABLE statement from the column types pandas infers
    df = pd.read_csv(csv_file)
    type_map = {"int": "INTEGER", "float": "FLOAT", "datetime": "TIMESTAMP"}
    cols = []
    for name, dtype in df.dtypes.items():
        sql_type = next((v for k, v in type_map.items() if k in str(dtype)), "STRING")
        cols.append(f"    {name} {sql_type}")
    return f"CREATE OR REPLACE TABLE {table_name} (\n" + ",\n".join(cols) + "\n)"

SUPPLIER_TABLE = "supplier_case_table"
cs.execute(create_table_sql(SUPPLIER_CSV, SUPPLIER_TABLE))
cs.execute(f"PUT 'file://{SUPPLIER_CSV.resolve()}' @%{SUPPLIER_TABLE}")
cs.execute(f'''
COPY INTO {SUPPLIER_TABLE}
FROM @%{SUPPLIER_TABLE}
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
''')
query_df(f"SELECT SupplierID, SupplierName, PostalPostalCode FROM {SUPPLIER_TABLE} LIMIT 5")

## 7. Weather for each supplier location

**7a. ZIP code centroids.** The 2021 Census Gazetteer gives the latitude and longitude of every ZIP Code Tabulation Area.

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE gaz_zcta (
    GEOID        STRING,
    ALAND        BIGINT,
    AWATER       BIGINT,
    ALAND_SQMI   FLOAT,
    AWATER_SQMI  FLOAT,
    INTPTLAT     FLOAT,
    INTPTLONG    FLOAT
)
''')
cs.execute(f"PUT 'file://{GAZ_FILE.resolve()}' @~/ OVERWRITE = TRUE")
cs.execute(f'''
COPY INTO gaz_zcta
FROM @~/{GAZ_FILE.name}
FILE_FORMAT = (TYPE = 'CSV' FIELD_DELIMITER = '\t' SKIP_HEADER = 1 TRIM_SPACE = TRUE)
''')
query_df("SELECT COUNT(*) AS zip_codes FROM gaz_zcta")

**7b. Nearest weather station.** For every supplier ZIP code, `ST_DISTANCE` finds the closest NOAA station within 50 km. The station index comes from the free Cybersyn *Weather & Environment* listing on the Snowflake Marketplace, mounted as the `WEATHER__ENVIRONMENT` database.

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE zip_weather_station AS
SELECT
    gz.GEOID                     AS ZIPCode,
    st.noaa_weather_station_id   AS WeatherStationID,
    ST_DISTANCE(ST_MAKEPOINT(gz.INTPTLONG, gz.INTPTLAT),
                ST_MAKEPOINT(st.longitude, st.latitude)) AS DistanceMeters
FROM (SELECT DISTINCT LPAD(PostalPostalCode::STRING, 5, '0') AS zip
      FROM supplier_case_table) sup
JOIN gaz_zcta gz
  ON sup.zip = gz.GEOID
CROSS JOIN WEATHER__ENVIRONMENT.CYBERSYN.NOAA_WEATHER_STATION_INDEX st
WHERE ST_DISTANCE(ST_MAKEPOINT(gz.INTPTLONG, gz.INTPTLAT),
                  ST_MAKEPOINT(st.longitude, st.latitude)) < 50000
QUALIFY ROW_NUMBER() OVER (PARTITION BY gz.GEOID ORDER BY DistanceMeters) = 1
''')
query_df("SELECT * FROM zip_weather_station ORDER BY ZIPCode")

**7c. Daily high temperature per supplier ZIP code.**

*Fix from the original version:* the earlier query joined weather by ZIP name instead of using the nearest station found above, and took the maximum of *every* weather variable (precipitation, snow and so on). This version uses the matched station and keeps only the maximum-temperature readings.

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE supplier_zip_code_weather AS
SELECT
    z.ZIPCode,
    t.date        AS WeatherDate,
    MAX(t.value)  AS DailyHighTemperature
FROM zip_weather_station z
JOIN WEATHER__ENVIRONMENT.CYBERSYN.NOAA_WEATHER_METRICS_TIMESERIES t
  ON t.noaa_weather_station_id = z.WeatherStationID
WHERE t.variable_name = 'Maximum Temperature'
GROUP BY z.ZIPCode, t.date
''')
query_df("SELECT * FROM supplier_zip_code_weather ORDER BY ZIPCode, WeatherDate LIMIT 10")

## 8. Final analysis table

Each invoice is joined to its supplier's location and to the daily high temperature on the transaction date.

*Improvement on the original version:* the earlier final table kept only ZIP code, date and temperature. It now keeps the supplier, invoice and invoiced-vs-quoted fields, so it can answer business questions directly.

In [ ]:
cs.execute('''
CREATE OR REPLACE TABLE final_table AS
SELECT
    poi.PurchaseOrderID,
    poi.SupplierID,
    sc.SupplierName,
    poi.SupplierTransactionID,
    TRY_TO_DATE(poi.TransactionDate) AS TransactionDate,
    poi.POAmount,
    poi.AmountExcludingTax,
    poi.Invoice_VS_Quoted,
    w.ZIPCode,
    w.DailyHighTemperature
FROM purchase_orders_and_invoices poi
JOIN supplier_case_table sc
  ON poi.SupplierID = sc.SupplierID
JOIN supplier_zip_code_weather w
  ON w.ZIPCode = LPAD(sc.PostalPostalCode::STRING, 5, '0')
 AND w.WeatherDate = TRY_TO_DATE(poi.TransactionDate)
''')
final = query_df("SELECT * FROM final_table")
print(f"{len(final):,} invoices with weather attached")
final.head()

## 9. Analysis: invoice accuracy and weather

With everything in one table, the pipeline can answer two questions a procurement team would ask:

1. How often do supplier invoices differ from what was quoted, and which suppliers are responsible?
2. Is there any relationship between local weather and invoice discrepancies?

In [ ]:
recon = query_df('''
SELECT
    s.SupplierName,
    COUNT(*)                                        AS invoices,
    SUM(IFF(ABS(p.Invoice_VS_Quoted) > 0.01, 1, 0)) AS invoices_with_discrepancy,
    ROUND(SUM(p.Invoice_VS_Quoted), 2)              AS net_over_under_billed,
    ROUND(SUM(p.POAmount), 2)                       AS total_po_value
FROM purchase_orders_and_invoices p
JOIN supplier_case_table s ON p.SupplierID = s.SupplierID
GROUP BY s.SupplierName
ORDER BY invoices_with_discrepancy DESC, total_po_value DESC
''')
recon.columns = recon.columns.str.lower()
recon["discrepancy_rate"] = recon["invoices_with_discrepancy"] / recon["invoices"]

total = recon["invoices"].sum()
flagged = recon["invoices_with_discrepancy"].sum()
print(f"{flagged:,} of {total:,} invoices ({flagged / total:.1%}) differ from the purchase-order amount")
recon

In [ ]:
df = final.copy()
df.columns = df.columns.str.lower()
df["has_discrepancy"] = df["invoice_vs_quoted"].abs() > 0.01
df["temp_band"] = pd.qcut(df["dailyhightemperature"], q=4,
                          labels=["Coolest 25%", "Cool", "Warm", "Hottest 25%"])

by_band = (df.groupby("temp_band", observed=True)
             .agg(invoices=("has_discrepancy", "size"),
                  discrepancy_rate=("has_discrepancy", "mean"),
                  avg_po_value=("poamount", "mean"))
             .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(by_band["temp_band"].astype(str), by_band["avg_po_value"], color="#4C72B0")
axes[0].set_title("Average purchase-order value by temperature band")
axes[0].set_ylabel("Average PO value ($)")

axes[1].bar(by_band["temp_band"].astype(str), by_band["discrepancy_rate"] * 100, color="#DD8452")
axes[1].set_title("Share of invoices that differ from quote")
axes[1].set_ylabel("% of invoices")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

by_band

In [ ]:
cs.close()
conn.close()